# 01 - Ingesta de Datos desde Roboflow (Bronze)

Este notebook descarga el dataset de deteccion de juguetes desde Roboflow,
lo almacena en un Volumen de Unity Catalog y crea la tabla **Bronze** con
las anotaciones YOLO parseadas.

**Salidas:**
- Imagenes y anotaciones en el Volume de Unity Catalog
- Tabla Bronze: `jgworkspaceclassic_catalog.infra_demo.cv_v2_bronze`

In [0]:
%pip install roboflow -q

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import os
import shutil
import yaml
from roboflow import Roboflow
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    ArrayType, FloatType, TimestampType
)

# -- Roboflow --
ROBOFLOW_API_KEY = ""
ROBOFLOW_WORKSPACE = ""
ROBOFLOW_PROJECT = ""
ROBOFLOW_VERSION = 2
ROBOFLOW_FORMAT = "yolo26"

# -- Unity Catalog --
CATALOG = "jgworkspaceclassic_catalog"
SCHEMA = "infra_demo"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/computer_vision"
LOCAL_DOWNLOAD = "/tmp/roboflow_dataset_v2"

# -- Tablas --
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.cv_v2_bronze"

print(f"Catalog: {CATALOG}")
print(f"Schema:  {SCHEMA}")
print(f"Volume:  {VOLUME_PATH}")
print(f"Bronze:  {BRONZE_TABLE}")

Catalog: jgworkspaceclassic_catalog
Schema:  infra_demo
Volume:  /Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision
Bronze:  jgworkspaceclassic_catalog.infra_demo.cv_v2_bronze


In [0]:
%sql
-- Verificar que el catalogo y esquema existen
USE CATALOG jgworkspaceclassic_catalog;
USE SCHEMA infra_demo;
SELECT current_catalog(), current_schema();

current_catalog(),current_schema()
jgworkspaceclassic_catalog,infra_demo


In [0]:
# Limpiar descarga previa si existe
if os.path.exists(LOCAL_DOWNLOAD):
    shutil.rmtree(LOCAL_DOWNLOAD)

# Descargar a /tmp (Roboflow usa rename/symlink no soportados por FUSE de Volumes)
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(ROBOFLOW_VERSION)
dataset = version.download(ROBOFLOW_FORMAT, location=LOCAL_DOWNLOAD)

# Copiar al volumen de UC
shutil.copytree(LOCAL_DOWNLOAD, VOLUME_PATH, dirs_exist_ok=True)

print(f"Dataset descargado en: {VOLUME_PATH}")
for item in sorted(os.listdir(VOLUME_PATH)):
    print(f"  {item}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /tmp/roboflow_dataset_v2 in yolo26:: 100%|██████████| 355/355 [00:00<00:00, 9871.76it/s]


Dataset descargado en: /Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision
  README.dataset.txt
  README.roboflow.txt
  data.yaml
  models
  test
  train
  valid


In [0]:
# Mostrar estructura de directorios y contar archivos
for split in ["train", "valid", "test"]:
    images_dir = os.path.join(VOLUME_PATH, split, "images")
    labels_dir = os.path.join(VOLUME_PATH, split, "labels")
    n_images = len(os.listdir(images_dir)) if os.path.exists(images_dir) else 0
    n_labels = len(os.listdir(labels_dir)) if os.path.exists(labels_dir) else 0
    print(f"  {split}: {n_images} imagenes, {n_labels} labels")

# Leer data.yaml
with open(os.path.join(VOLUME_PATH, "data.yaml")) as f:
    data_config = yaml.safe_load(f)

class_names = data_config["names"]
print(f"\nClases detectadas: {class_names}")
print(f"Numero de clases: {len(class_names)}")

  train: 153 imagenes, 153 labels
  valid: 15 imagenes, 15 labels
  test: 7 imagenes, 7 labels

Clases detectadas: ['Captain', 'Gohan']
Numero de clases: 2


In [0]:
rows = []
for split in ["train", "valid", "test"]:
    labels_dir = os.path.join(VOLUME_PATH, split, "labels")
    images_dir = os.path.join(VOLUME_PATH, split, "images")

    if not os.path.exists(labels_dir):
        print(f"  WARN: {labels_dir} no existe, saltando...")
        continue

    for label_file in sorted(os.listdir(labels_dir)):
        if not label_file.endswith(".txt"):
            continue

        image_id = os.path.splitext(label_file)[0]

        # Buscar la imagen correspondiente
        image_path = None
        for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
            candidate = os.path.join(images_dir, image_id + ext)
            if os.path.exists(candidate):
                image_path = candidate
                break

        # Obtener tamano de la imagen en bytes
        image_size_bytes = os.path.getsize(image_path) if image_path else 0

        label_path = os.path.join(labels_dir, label_file)
        with open(label_path) as f:
            lines = f.readlines()

        for obj_idx, line in enumerate(lines):
            parts = line.strip().split()
            if len(parts) < 3:
                continue

            class_id = int(parts[0])
            coords = [float(x) for x in parts[1:]]
            num_points = len(coords) // 2

            # Bounding box del poligono
            xs = coords[0::2]
            ys = coords[1::2]

            rows.append((
                image_id,
                image_path,
                label_path,
                split,
                class_id,
                class_names[class_id] if class_id < len(class_names) else "unknown",
                obj_idx,
                coords,
                num_points,
                float(min(xs)),
                float(min(ys)),
                float(max(xs)),
                float(max(ys)),
                image_size_bytes,
                len(lines),  # total objetos en esta imagen
            ))

print(f"Total anotaciones parseadas: {len(rows)}")

Total anotaciones parseadas: 189


In [0]:
schema = StructType([
    StructField("image_id", StringType(), False),
    StructField("image_path", StringType(), True),
    StructField("label_path", StringType(), False),
    StructField("split", StringType(), False),
    StructField("class_id", IntegerType(), False),
    StructField("class_name", StringType(), False),
    StructField("object_index", IntegerType(), False),
    StructField("polygon_coords", ArrayType(FloatType()), False),
    StructField("num_polygon_points", IntegerType(), False),
    StructField("bbox_x_min", FloatType(), False),
    StructField("bbox_y_min", FloatType(), False),
    StructField("bbox_x_max", FloatType(), False),
    StructField("bbox_y_max", FloatType(), False),
    StructField("image_size_bytes", IntegerType(), False),
    StructField("objects_in_image", IntegerType(), False),
])

df_bronze = spark.createDataFrame(rows, schema=schema)

# Agregar metadata de ingesta
df_bronze = (
    df_bronze
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("source", F.lit("roboflow"))
    .withColumn("roboflow_project", F.lit(ROBOFLOW_PROJECT))
    .withColumn("roboflow_version", F.lit(ROBOFLOW_VERSION))
)

# Guardar como tabla Delta
df_bronze.write.mode("overwrite").saveAsTable(BRONZE_TABLE)

print(f"Tabla Bronze guardada: {BRONZE_TABLE}")
print(f"Total registros: {spark.table(BRONZE_TABLE).count()}")

Tabla Bronze guardada: jgworkspaceclassic_catalog.infra_demo.cv_v2_bronze
Total registros: 189


In [0]:
display(spark.table(BRONZE_TABLE).limit(10))

image_id image_path label_path split class_id class_name object_index polygon_coords num_polygon_points bbox_x_min bbox_y_min bbox_x_max bbox_y_max image_size_bytes objects_in_image ingested_at source roboflow_project roboflow_version IMG_20260421_072248609_jpg.rf.5fed3897b36adc346310e2ad5075c146 /Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/train/images/IMG_20260421_072248609_jpg.rf.5fed3897b36adc346310e2ad5075c146.jpg /Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/train/labels/IMG_20260421_072248609_jpg.rf.5fed3897b36adc346310e2ad5075c146.txt train 0 Captain 0 List(0.5046875, 0.2359375, 0.5109375, 0.2390625, 0.5359375, 0.2390625, 0.559375, 0.24375, 0.5921875, 0.246875, 0.6, 0.2515625, 0.6171875, 0.2546875, 0.6328125, 0.2609375, 0.6703125, 0.2859375, 0.684375, 0.3, 0.696875, 0.325, 0.6953125, 0.3265625, 0.69375, 0.365625, 0.690625, 0.371875, 0.690625, 0.3890625, 0.6921875, 0.390625, 0.690625, 0.4140625, 0.6875, 0.41875, 0.6859375, 0.4390625, 0.675, 0.471875, 0.6609375, 0.4953125, 0.6515625, 0.5046875, 0.6515625, 0.5109375, 0.665625, 0.5296875, 0.665625, 0.5375, 0.6578125, 0.5421875, 0.6453125, 0.54375, 0.634375, 0.5515625, 0.6328125, 0.55, 0.6140625, 0.55, 0.59375, 0.5421875, 0.5875, 0.5359375, 0.5859375, 0.528125, 0.5828125, 0.525, 0.5453125, 0.525, 0.5421875, 0.5234375, 0.5265625, 0.5390625, 0.51875, 0.5421875, 0.50625, 0.55625, 0.509375, 0.5609375, 0.521875, 0.5625, 0.5265625, 0.565625, 0.5328125, 0.575, 0.55625, 0.565625, 0.5703125, 0.5640625, 0.571875, 0.56875, 0.5703125, 0.5796875, 0.5578125, 0.60625, 0.5578125, 0.61875, 0.5546875, 0.6296875, 0.5578125, 0.6328125, 0.565625, 0.63125, 0.5703125, 0.6359375, 0.5703125, 0.659375, 0.5765625, 0.6671875, 0.5796875, 0.6765625, 0.575, 0.6875, 0.5765625, 0.69375, 0.5984375, 0.6890625, 0.625, 0.671875, 0.6296875, 0.671875, 0.6328125, 0.6765625, 0.628125, 0.6859375, 0.6046875, 0.7140625, 0.6015625, 0.721875, 0.5921875, 0.7296875, 0.54375, 0.7390625, 0.5359375, 0.7375, 0.5015625, 0.74375, 0.4953125, 0.7421875, 0.475, 0.74375, 0.45625, 0.7484375, 0.446875, 0.7484375, 0.43125, 0.75625, 0.4, 0.7578125, 0.3546875, 0.7703125, 0.3171875, 0.7671875, 0.3140625, 0.7640625, 0.303125, 0.7625, 0.3046875, 0.7546875, 0.321875, 0.7375, 0.321875, 0.7328125, 0.3046875, 0.725, 0.2828125, 0.71875, 0.2640625, 0.6953125, 0.2484375, 0.6875, 0.246875, 0.6859375, 0.2484375, 0.6796875, 0.2421875, 0.6625, 0.23125, 0.653125, 0.2265625, 0.6453125, 0.2265625, 0.634375, 0.2296875, 0.63125, 0.2375, 0.6296875, 0.2421875, 0.625, 0.23125, 0.6125, 0.2265625, 0.6015625, 0.228125, 0.5984375, 0.2296875, 0.596875, 0.25, 0.5984375, 0.2640625, 0.5796875, 0.2734375, 0.5609375, 0.2796875, 0.5546875, 0.2796875, 0.5515625, 0.29375, 0.5453125, 0.303125, 0.5453125, 0.3078125, 0.540625, 0.31875, 0.5359375, 0.321875, 0.5375, 0.3328125, 0.5359375, 0.3640625, 0.51875, 0.3671875, 0.5203125, 0.375, 0.5140625, 0.3625, 0.5078125, 0.3390625, 0.5046875, 0.321875, 0.4984375, 0.2953125, 0.4796875, 0.2859375, 0.4640625, 0.2828125, 0.45, 0.2765625, 0.4359375, 0.2765625, 0.4265625, 0.2734375, 0.41875, 0.275, 0.3671875, 0.2828125, 0.340625, 0.2828125, 0.3171875, 0.2859375, 0.296875, 0.296875, 0.278125, 0.315625, 0.2625, 0.3625, 0.240625, 0.4140625, 0.2328125, 0.4171875, 0.234375, 0.503125, 0.2359375) 132 0.2265625 0.2328125 0.696875 0.7703125 28305 1 2026-04-23T01:43:58.882Z roboflow toys-detection-bzeh7 2 IMG_20260421_072248609_jpg.rf.a245f43d4c045de92ce7869f9f9907de /Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/train/images/IMG_20260421_072248609_jpg.rf.a245f43d4c045de92ce7869f9f9907de.jpg /Volumes/jgworkspaceclassic_catalog/infra_demo/computer_vision/train/labels/IMG_20260421_072248609_jpg.rf.a245f43d4c045de92ce7869f9f9907de.txt train 0 Captain 0 List(0.425, 0.2484375, 0.4265625, 0.25, 0.4390625, 0.2484375, 0.446875, 0.2515625, 0.4515625, 0.25, 0.4671875, 0.2546875, 0.4734375, 0.253125, 0.534375, 0.2671875, 0.5421875, 0.271875, 0.5734375, 0.278125, 0.5828125, 0.2828125, 0.59375, 0.284375, 0.59

In [0]:
df = spark.table(BRONZE_TABLE)
print("=" * 55)
print("RESUMEN DE INGESTA BRONZE")
print("=" * 55)
print(f"  Total anotaciones: {df.count()}")
print(f"  Imagenes unicas:   {df.select('image_id').distinct().count()}")
print(f"  Clases:            {[r.class_name for r in df.select('class_name').distinct().collect()]}")

display(
    df.groupBy("split", "class_name")
    .agg(
        F.count("*").alias("anotaciones"),
        F.countDistinct("image_id").alias("imagenes")
    )
    .orderBy("split", "class_name")
)

RESUMEN DE INGESTA BRONZE
  Total anotaciones: 189
  Imagenes unicas:   175
  Clases:            ['Captain', 'Gohan']


split,class_name,anotaciones,imagenes
test,Captain,2,2
test,Gohan,6,6
train,Captain,108,108
train,Gohan,57,57
valid,Captain,10,10
valid,Gohan,6,6
